# 5. GridSearchCV - Hyperparameter Tuning

In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score

In [11]:
dataset1=pd.read_csv("phishing_preprocessed.csv")

df2=dataset1

indep_X=df2.drop('status',axis=1)
dep_Y=df2['status']

## 5.1 Decision Tree – SelectKBest Feature Selection

In [3]:
from sklearn.feature_selection import SelectKBest,chi2

X_train,X_test,y_train,y_test=train_test_split(indep_X,dep_Y,test_size=0.25,random_state=0,stratify=dep_Y) 

sc=MinMaxScaler()
X_train_scaled=sc.fit_transform(X_train)
X_test_scaled=sc.transform(X_test)

kbest=SelectKBest(score_func=chi2,k=4)
X_train_kbest=kbest.fit_transform(X_train_scaled,y_train)
X_test_kbest=kbest.transform(X_test_scaled)

## 5.2 Decision Tree – GridSearchCV

In [4]:
from sklearn.tree import DecisionTreeClassifier
param_grid={'criterion':['gini','entropy'],'max_features':['sqrt','log2'],'splitter':['best','random']}
grid= GridSearchCV(DecisionTreeClassifier(random_state=0),param_grid,
                   refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
grid.fit(X_train_kbest,y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


,estimator,DecisionTreeC...andom_state=0)
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_features': ['sqrt', 'log2'], 'splitter': ['best', 'random']}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [5]:
print("Best CV Score:",grid.best_score_)

Best CV Score: 0.9134338381608675


In [6]:
y_pred=grid.predict(X_test_kbest)

In [7]:
cm=confusion_matrix(y_test,y_pred)
print(cm)

[[1298  131]
 [ 113 1316]]


In [8]:
clf_report=classification_report(y_test,y_pred)
print(clf_report)

              precision    recall  f1-score   support

           0       0.92      0.91      0.91      1429
           1       0.91      0.92      0.92      1429

    accuracy                           0.91      2858
   macro avg       0.91      0.91      0.91      2858
weighted avg       0.91      0.91      0.91      2858



In [9]:
roc_auc_score(y_test,grid.predict_proba(X_test_kbest)[:,1])

0.9642396993987878

In [10]:
re=grid.cv_results_
table=pd.DataFrame.from_dict(re)
display(table)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.011188,0.002040,0.011071,0.000911,gini,sqrt,best,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.90903,0.911953,0.915394,0.918311,0.910733,0.913084,0.003343,3
1,0.008554,0.002044,0.009953,0.001180,gini,sqrt,random,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.90903,0.911953,0.916559,0.918311,0.911316,0.913434,0.003453,1
2,0.009116,0.000973,0.010987,0.001969,gini,log2,best,"{'criterion': 'gini', 'max_features': 'log2', ...",0.90903,0.911953,0.915394,0.918311,0.910733,0.913084,0.003343,3
3,0.008142,0.001007,0.012307,0.000762,gini,log2,random,"{'criterion': 'gini', 'max_features': 'log2', ...",0.90903,0.911953,0.916559,0.918311,0.911316,0.913434,0.003453,1
4,0.009522,0.002307,0.011891,0.002231,entropy,sqrt,best,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.90903,0.911953,0.915394,0.918311,0.910733,0.913084,0.003343,3
5,0.009840,0.000746,0.009722,0.001356,entropy,sqrt,random,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.90903,0.911953,0.915394,0.917726,0.911316,0.913084,0.003089,7
6,0.009520,0.001395,0.012273,0.001489,entropy,log2,best,"{'criterion': 'entropy', 'max_features': 'log2...",0.90903,0.911953,0.915394,0.918311,0.910733,0.913084,0.003343,3
7,0.008701,0.001963,0.009415,0.001516,entropy,log2,random,"{'criterion': 'entropy', 'max_features': 'log2...",0.90903,0.911953,0.915394,0.917726,0.911316,0.913084,0.003089,7


### 5.3 KNN – RFE Feature Selection

In [12]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

X_train,X_test,y_train,y_test=train_test_split(indep_X,dep_Y,test_size=0.25,random_state=0,stratify=dep_Y)

log_model=LogisticRegression(solver='liblinear',max_iter=1000,random_state=0)
rfe=RFE(estimator=log_model,n_features_to_select=6,step=20)

X_train_rfe=rfe.fit_transform(X_train,y_train)
X_test_rfe=rfe.transform(X_test)

In [13]:
sc=StandardScaler()
X_train_scaled=sc.fit_transform(X_train_rfe)
X_test_scaled=sc.transform(X_test_rfe)

### 5.4 KNN - GridSearchCV

In [14]:
from sklearn.neighbors import KNeighborsClassifier
param_grid={'n_neighbors':[3,5,7,9,11],'metric':['minkowski'],'p':[1,2]}
grid=GridSearchCV(KNeighborsClassifier(),param_grid,
                   refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
grid.fit(X_train_scaled,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,estimator,KNeighborsClassifier()
,param_grid,"{'metric': ['minkowski'], 'n_neighbors': [3, 5, ...], 'p': [1, 2]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,9


In [15]:
print("Best CV Score:",grid.best_score_)

Best CV Score: 0.8727930593102846


In [16]:
y_pred=grid.predict(X_test_scaled)

In [17]:
cm=confusion_matrix(y_test,y_pred)
print(cm)

[[1175  254]
 [ 122 1307]]


In [18]:
clf_report=classification_report(y_test,y_pred)
print(clf_report)

              precision    recall  f1-score   support

           0       0.91      0.82      0.86      1429
           1       0.84      0.91      0.87      1429

    accuracy                           0.87      2858
   macro avg       0.87      0.87      0.87      2858
weighted avg       0.87      0.87      0.87      2858



In [19]:
roc_auc_score(y_test,grid.predict_proba(X_test_scaled)[:,1])

0.9276779947121532

In [20]:
re=grid.cv_results_
table=pd.DataFrame.from_dict(re)
display(table)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_metric,param_n_neighbors,param_p,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.024682,0.004666,0.429780,0.021625,minkowski,3,1,"{'metric': 'minkowski', 'n_neighbors': 3, 'p': 1}",0.874423,0.870160,0.864812,0.877199,0.872575,0.871834,0.004199,5
1,0.014894,0.001233,0.406335,0.046663,minkowski,3,2,"{'metric': 'minkowski', 'n_neighbors': 3, 'p': 2}",0.874423,0.870160,0.864812,0.877199,0.872575,0.871834,0.004199,5
2,0.014496,0.001257,0.380645,0.005271,minkowski,5,1,"{'metric': 'minkowski', 'n_neighbors': 5, 'p': 1}",0.874435,0.869583,0.863659,0.877790,0.872600,0.871613,0.004785,10
3,0.014100,0.000918,0.371049,0.006704,minkowski,5,2,"{'metric': 'minkowski', 'n_neighbors': 5, 'p': 2}",0.874435,0.870160,0.863659,0.877790,0.872600,0.871729,0.004741,9
4,0.014900,0.001203,0.438066,0.010367,minkowski,7,1,"{'metric': 'minkowski', 'n_neighbors': 7, 'p': 1}",0.874435,0.871906,0.863083,0.877790,0.871443,0.871731,0.004877,7
5,0.015205,0.001217,0.427969,0.004767,minkowski,7,2,"{'metric': 'minkowski', 'n_neighbors': 7, 'p': 2}",0.874435,0.871906,0.863083,0.877790,0.871443,0.871731,0.004877,7
6,0.017198,0.001595,0.477481,0.009043,minkowski,9,1,"{'metric': 'minkowski', 'n_neighbors': 9, 'p': 1}",0.875013,0.872515,0.865456,0.877790,0.873190,0.872793,0.004098,1
7,0.017793,0.002406,0.445894,0.007000,minkowski,9,2,"{'metric': 'minkowski', 'n_neighbors': 9, 'p': 2}",0.875013,0.872515,0.865456,0.877790,0.872600,0.872675,0.004093,2
8,0.016819,0.001275,0.482984,0.006516,minkowski,11,1,"{'metric': 'minkowski', 'n_neighbors': 11, 'p'...",0.875013,0.870753,0.864879,0.877790,0.872009,0.872089,0.004355,3
9,0.016818,0.001639,0.407725,0.055872,minkowski,11,2,"{'metric': 'minkowski', 'n_neighbors': 11, 'p'...",0.875013,0.870753,0.864879,0.877790,0.872009,0.872089,0.004355,3
